# Введение

Видеоданные играют огромную роль в современном мире. Согласно статистике, предоставленной Cisco, более 70% мирового интернет-трафика в 2023г. составляет передача видео. В этот трафик входят видеоконференции, потоковое вещание, облачные игры, а также сервисы видеохостинга, такие как YouTube, камеры видеонаблюдения и т.д.

<table>
    <tr>
        <td>
            <img src="https://cdn.statcdn.com/Infographic/images/normal/1620.jpeg" />
        </td>
        <td>
            <img src="https://xrdocs.io/design/images/vni_video_traffic.png" />
        </td>
    </tr>
</table> 

При этом среднее качество видео, передаваемого по сети, также растет. Современные сервисы предоставляют видео в формате 4К или даже 8К 60FPS. Без сжатия 1 минута видео 1080p может весить около 11 ГБ, которые нужно как-то передать по каналу с ограниченной пропускной способностью.

Поэтому чрезвычайно важно эффективно сжимать информацию такого типа. Для решения этой задачи разрабатываются и постоянно совершенствуются специальные программы -- видеокодеки, а также ворматы хранения сжатых данных.

В этой серии ноутбуков я предлагаю разобраться, на каких принципах основаны видео кодеки, как они работают, а также попробовать (практически, за исключением матпакетов) с нуля реализовать простейший прототип видео кодека. Несомненно, такой подход будет работать хуже по всем параметрам, чем современные решения, но в первую очередь этот материал нужен для того, чтобы составить общее представление о том, как такие решения в принципе работают. Начать же я предлагаю с того, как в принципе можно представить какие-то данные в виде нулей и единиц, и, желательно, не потратить все доступное место на жестких дисках мира для сохранения какой-нибудь маленькой картинки. Правда начать придется ОЧЕНЬ издалека, со школьной программы по информатике (не у меня одного же неравномерное кодирование было в школе, правда??).

# Кодирование данных

Вообще, даже для тех, кто далек от разработки, не является секретом, что все данные в памяти компьютера представляются в виде последовательностей нулей и единиц. Для экономии времени (ха-ха) предлагаю не останавливаться на том, какие существуют типы данных, какие их особенности и как именно представить их в виде последовательности нулей и единиц. Упомяну только, что существуют целые числа (integer, знаковые и беззнаковые) и числа с плавающей точкой, и все они в свою очередь могут быть представлены разным количеством бит, обычно 8, 16, 32 и иногда редко 64. Чем больше бит, тем более разнообразные данные можно закодировать в таком представлении.

Рассмотрим задачу: *В некотором алфавите есть всего 3 буквы: **A**, **B** и **C**. Все слова состоят из какой-то комбинации этих 3 букв. Сколько нужно бит информации, чтобы закодировать слово длины N, состоящее из букв этого алфавита.*

Давайте присвоим каждой букве уникальное число. Пусть букве **А** будет соответствовать число **0**, **B** -- **1**, **C** -- **2**. Представим эти числа в двоичной системе счисления: **А**=***0***, **B**=***1***, **C**=***10***. Разумеется, в таком виде числа оставить нельзя, так как сразу видно, что для кода `10` нельзя однозначно определить, соответствует ли такому коду последовательность **BA** или **C**. Поэтому для определенности дополним все числа вначале незначащими битами до максимальной длины:

<img src="data/simple_coding.png" />

Таким образом на кодирование каждого символа уходит 2 бита информации, значит на кодирование полного слова уйдет $N*2$ бит. Все просто! Вот только сразу возникает вопрос, а можно ли лучше? И ответ, разумеется, "да, можно". В одной из ветвей дерева нет значения, а значит для символа **C** код можно сократить до ***1***. При этом код все еще можно будет однозначно декодировать. для этого будем последовательно читать символы и строить по дереву путь до конкретного листа. При попадании в листовой элемент можно определить соответствующий символ, а со следующего бита начинать декодирование следующего символа:

<img src="data/decoding.png" />

В последовательности на примере нам удалось сократить длину строки на 3 бита, ровно столько в ней встречается символ **C**. Осталось оценить количество бит, необходимых для кодирования строки длины N. Но тут есть проблема: мы не знаем, сколько именно в данной последовательности будет букв **С**. Если слово будет состоять целиком из этих букв, то последовательность будет в 2 раза короче, чем если слово не будет содержать этой буквы вообще. Таким образом точное значение будет находиться где-то между $N$ и $2*N$.

Предположим, что в последовательности букв в слове, нет никакой закономерности, каждая следующая буква может с равной вероятностью быть **A**, **B** или **С**. Тогда с вероятностью $2/3$ буква будет занимать 2 бита, а с вероятностью $1/3$ только 1 бит. В таком случае можно сказать, что **в среднем** 1 буква занимает $2*0.667 + 1*0.333=1.667$ бит, а слово длины $N$, соответственно, $1.667*N$ бит.

Кажется, удалось немного сократить. Но можно ли лучше?

# Энтропийное кодирование

А тут ответ не так однозначен, как в прошлый раз. Вернемся к оценке количества бит на одну букву из предыдущего примера. Для вычисления мы сделали предположение, что каждая буква может встретиться с равной вероятностью. Но если вероятности не равны, то оценка будет смещаться в сторону увеличения, если буква **C** будет встречаться реже, чем в $1/3$ случаев или наоборот в сторону увеличения, если она будет встречаться чаще. А образом наша оченка количества бит, которые необходимы для кодирования буквы -- это всего лишь **математическое ожидание дискретной случайной величины $x$, принимающей значение 1 с вероятностью $P(x=1)$ и 2 с вероятностью $1-P(x=1)$**. Осталось только каким-то образом применить это знание для уменьшения количества информации при кодировании.

В предыдущей задаче, если вероятности не равны, очевидным решением будет присвоить самый короткий код символу, который чаще всего встречается. Как видно из формулы, это позволит уменьшить количество информации. Посмотрим, как такой подход обобщается на более сложный случай -- добавим в наш алфавит букву **D** и построим новое дeрево:

<img src="data/eq_tree.png" />

При таком подходе любой символ равномерно кодируется 2 битами данных, так что матожидание так же будет равно 2. Но это не единственный способ представить дерево с 4 листьями. Рассмотрим другое представление такого дерева и попытаемся определить, является ли такое представление более эффективным, и в каких случаях.

<img src="data/neq_tree.png" />

Заметим, что  в таком представлении некоторые буквы будут кодироваться 3 битами. Мы уже договорились, что более часто встречающиеся буквы стоит кодировать более коротким кодом, а более редкие -- более длинным. Если вероятности буквы будут равны, то такое представление только ухудшит ситуацию. Интуитивно это можно заметить, так как ради того, чтобы кодировать 1 букву в 1 бит, нам пришлось присвоить код из 3 бит целым 2 буквам. Но интуитивному пониманию верить нельзя, так что вот расчет:

$$0.25*1+0.25*2+0.25*3+0.25*3=2.25 > 2$$

Ситуация существенно меняется, если вероятности встретить конкретную букву будут не одинаковыми, а существенно отличаться. Пусть $P(x=A)=0.125$, $P(x=B)=0.125$, $P(x=C)=0.25$, $P(x=D)=0.5$. Пока не важно, откуда я взял эти числа, но легко посчитать матожидание количества бит на одну букву по предыдущей формуле:

$$0.125*3 + 0.125*3 + 0.25*2 + 0.5*1 = 1.75 < 2$$

В таком сценарии ситуация существенно изменилась: теперь количество информации для кодирования 1 символа уменьшилось по сравнению с симметричным деревом.

Итак, количество информации, которое требуется для кодирования, зависит от вероятности встретить каждый конкретный символ в последовательности и того, как составлено само дерево. При этом нам хотелось бы, чтобы дерево было составлено так, чтобы значение нашей оценки количества информации было минимальным среди всех возможных вариантов такого дерева. Для этого при построении мы также будем использовать вероятности встретить конкретный символ.

Рассмотрим алгоритм построения дерева для алфавита с заданными вероятностями встретить каждый конкретный символ
1) Инициализируем листья дерева нашими вероятностями символов из алфавита и помещаем их в общий список
2) Выбираем 2 элемента с **наименьшими** вероятностями и извлекаем их из списка
3) Создаем узел дерева, потомками которого являются элементы, выбранные на предыдущем шаге. Вероятностью попасть в этот узел будет сумма вероятностей извлеченных элементов. Помещаем этот элемент в список всех узлов.
4) Если в результате этих операций в списке осталось больше 1 элемента, возвращаемся к п.2
5) При поиске по полученному двоичному дереву если мы переходим в левого потомка,  добавляем к коду символа 0, иначе 1.
6) Поздравляю, вы великолепны!

Простая реализация "в лоб" будет работать достаточно медленно, для использования на практике, когда существуют сотни или даже тысячи элементов, которые необходимо кодировать, стоит использовать, например, кучу или какую-то еще реализацию очереди с приоритетами. Однако для нашего модельного примера этого будет более чем достаточно, перейдем к реализации.

In [17]:
# Создадим класс, описывающий элемент дерева
# У каждого элемента задается вероятность, а также может быть значение (в случае, если это лист) или потомки (если это узел)

class Node:
    def __init__(self, prob, left=None, right=None, value=None):
        self.prob = prob
        self.left = left
        self.right = right
        self.value = value

    def get_children(self):
        return self.left, self.right

    def get_value(self):
        return self.value

    def get_prob(self):
        return self.prob

In [18]:
# Напишем функцию построения дерева по спискам вероятностей и символов

def get_tree(probs, values):
    # Создаем список листьев дерева из элементов
    nodes = [Node(p, value=v) for p, v in zip(probs, values)]
    while len(nodes) > 1:
        # Сортируем элементы списка по вероятности
        nodes.sort(key=lambda x: x.get_prob())
        # Выбираем первые 2 элемента списка
        first = nodes.pop(0)
        second = nodes.pop(0)
        # Вычисляем вероятность для общего узла
        new_prob = first.get_prob() + second.get_prob()
        # Создаем общий узел и добавляем в список
        nodes.append(Node(new_prob, left=first, right=second))
    return nodes[0]

# Соберем словарь кодов символов для удобства кодирования и визуализации

def get_codes(tree):
    # Добавим корневой элемент в список и создадим пустой словарь для значений
    nodes = [("", tree)]
    values = {}
    # Пока список элементов не пустой
    for code, node in nodes:
        # Проверяем, является ли нода листом
        if node.get_value() is None:
            # Если не является, добавляем в список поиска потомков и их коды
            nodes.append((code + "0", node.left))
            nodes.append((code + "1", node.right))
        else:
            # Если является листом, добавляем значение и его код в словарь
            values[node.get_value()] = code
    return values

In [19]:
# Задалим наш алфавит и вероятности символов
probs = [0.12, 0.55, 0.12, 0.21]
values = ["A", "B", "C", "D"]

In [20]:
# Соберем дерево Хаффмана и преобразуем в словарь
tree = get_tree(probs, values)
codes = get_codes(tree)
codes

{'B': '1', 'D': '00', 'A': '010', 'C': '011'}

Итак, у нас есть двоичное дерево. Доказательством его оптимальности в этом материале предлагаю не заниматься, для этого существует большое количество отдельных ресурсов, где доказывают теоремы, разбирают эффективные реализации и т.д., а я предлагаю сразу перейти к интересному -- кодированию и декодированию.

С кодированием все просто: для каждой буквы из словаря берем соответствующий ей код и собираем в одну строку:

In [25]:
def encode_string(string, codes):
    bit_stream = ""
    for char in string:
        bit_stream += codes[char]
    return bit_stream

In [29]:
string = "BBDBADBCABBD"

bit_stream = encode_string(string, codes)
bit_stream

'110010100010110101100'

А вот для декодирования придется выполнить поиск по дереву. Для этого будем последовательно читать строку по 1 биту и в зависимости от значения переходить в левое или правое поддерево от текущего элемента

In [30]:
def decode_scring(bit_stream, tree):
    _curr = tree
    string = ""
    for bit in bit_stream:
        if bit == "0":
            _curr = _curr.left
        else:
            _curr = _curr.right
        if _curr.get_value() is not None:
            string += _curr.get_value()
            _curr = tree
    return string

In [31]:
decode_scring(bit_stream, tree)

'BBDBADBCABBD'

В данной случае следующая часть не является значимой, но предлагаю также для более честного представления результатов написать функции, которые будут сохранять полученный байткод в файл и загружать его обратно.

Так как сохраняемая последовательность может иметь произвольную длину, а кроме последовательности в файл не будет записываться дополнительной информации, вместо сохранения полной длины последовательности, можно сохранить остаток от деления длины строки на 8 (количество бит в байте), дополнить строку незначащими нулями до длины кратной 8 и сохранить, как последовательность байтов.

При декодировании считывается вся строка, и остаток, далее отделяются незначащие нули и производится декодирование.

In [38]:
def save_bytecode(binary_string, fname):
    # Вычисляем количество дополнительных бит
    num_padding_bits = 8 - (len(binary_string) % 8)
    # Добиваем строку нулями до длины кратной 8
    binary_string = "0" * num_padding_bits + binary_string

    # Собираем список байт в числах
    byte_values = [num_padding_bits]
    for i in range(0, len(binary_string), 8):
        byte_chunk = binary_string[i:i+8]
        integer_value = int(byte_chunk, 2)
        byte_values.append(integer_value)

    # Кодируем и сохраняем в файл
    bytecode = bytes(byte_values)
    with open(fname, "wb") as f:
        f.write(bytecode)

def load_bytecode(fname):
    with open(fname, "rb") as f:
        bytecode = f.read()

    # Переводим байткод в список чисел и сохраняем количество дополнительных бит
    byte_values = list(bytecode)
    num_padding_bits = byte_values[0]

    # Каждое число из списка переводим в двоичную строку
    binary_string = ""
    for i in byte_values[1:]:
        binary_string += "{:08b}".format(i)

    # Отрезаем дополнительные биты и возвращаем строку исходной длины
    return binary_string[num_padding_bits:]


Протестируем полный пайплайн с сохранением и загрузкой из файла

In [39]:
string = "BDABDAADADDBBA"

bin_string = encode_string(string, codes)
print("Binary string:", bin_string)

save_bytecode(bin_string, "test.bin")

bin_string_rec = load_bytecode("test.bin")
print("Recovered binary string:", bin_string_rec)

decoded = decode_scring(bin_string_rec, tree)
print(decoded)

Binary string: 10001010001001000010000011010
Recovered binary string: 10001010001001000010000011010
BDABDAADADDBBA


# Так при чем же здесь энтропия?


Вернемся немного назад. Когда у нас есть сообщение, которое хотелось бы передать по какому-то каналу связи, нам обычно хочется понимать, сколько бит информации для этого потребуется. А значит хотелось бы понимать, сколько информации в среднем будет нужно для кодирования одного символа. Как уже стало понятно из предыдущего раздела, если при кодировании учитывать информацию о частотности символов, то кодирование можно выполнить более эффективно. А есть ли теоретический минимум для количества бит, ниже которого уже не получится кодировать информацию без потерь?

По давно сложившейся традиции, если автор какого-то учебного материала задает подобный вопрос, ответ на него будет положительным. И ответ на этот вопрос даёт **энтропия Шеннона** -- мера информационной неопределённости или «степени неожиданности» источника данных.

Термин «энтропия» в теорию информации ввёл Клод Шеннон в 1948 году в своей основополагающей работе «Математическая теория связи». Шеннон заимствовал это понятие из термодинамики, где энтропия характеризует меру неопределённости или беспорядка в системе. По аналогии, энтропия Шеннона измеряет степень неопределённости, связанной со случайной величиной или источником данных. Джон фон Нейман, консультируя Шеннона, шутливо заметил: «Назови это энтропией — в спорах ты всегда будешь иметь преимущество, потому что никто не знает, что такое энтропия».

**Определение и вычисление**

Пусть дискретный источник генерирует символы из алфавита $A = {a_1, a_2, ..., a_n}$. Каждый символ $a_i$ появляется с известной вероятностью $p(a_i)$, где $0 ≤ p(a_i) ≤ 1$ и сумма всех вероятностей равна 1.

Энтропия Шеннона $H(A)$ этого источника измеряется в битах на символ и вычисляется по формуле:

$$
H(A) = - \sum_{i=1...n}\left(p(a_i) * \log_2p(a_i) \right)
$$

Интуиция: Слагаемое $-\log_2p(a_i)$ можно интерпретировать как «информационный вес» или «степень удивления» от появления символа $a_i$. Редкие символы (с малой вероятностью $p$) имеют большой информационный вес. Энтропия $H(A)$ — это среднее (математическое ожидание) информационного веса по всем символам алфавита.

Абстрактный базовый внимательный читатель может заметить, что очень похожую формулу уже встречали выше, когда считали среднее количество бит на символ в коде, а может даже заметил, что длина битовой последовательности как раз была равна $-\log_2p(a_i)$ (там для вероятностей я как раз обратные степени двойки брал, чтобы считать удобнее было).

**Ключевой результат, связывающий энтропию и сжатие данных, гласит:**

> Для любого префиксного (или любого однозначно декодируемого) кода средняя длина кода $L$ на символ не может быть меньше энтропии источника $H(A)$.
>
> Формально: $L ≥ H(A)$. При этом можно построить такие коды (например, код Хаффмана), что $L < H(A) + 1$.

Оставим, пожалуй, данный факт без строгого доказательства, ибо в противном случае я рискую вообще никогда не закончить этот материал.

**Свойства энтропии Шеннона**

У энтропии есть несколько полезных свойств, которые полезно знать, если приходится работать с кодированием информации:

1. $H(X) \geq 0$, причём равенство нулю достигается тогда и только тогда, когда одна из вероятностей равна $1$, а остальные — $0$ (источник детерминированный).
2. Максимум энтропии для алфавита из $n$ символов достигается при равномерном распределении $p_i = 1/n$ и равен $\log_2 n$.
3. Энтропия аддитивна для независимых источников.





# Выводы?

Итак, надеюсь теперь стало более понятно, как происходит кодирование информации, и мы даже можем как-то кодировать и декодировать текстовые строки в последовательности ноликов и единичек. У нас есть как готовая реализация алгоритма кодирования Хаффмана, так и теоретическая оценка на минимальную длину кода от Шеннона. Самая простая часть (школьный курс информатики, да) закончился, дальше надо будет разбираться, как применить полученные знания к кодированию картинок (и что вообще такое эти ваши изображения). Будем ~~сжимать шакалов~~ писать свой недожипег. Надеюсь, это произойдет чуть раньше чем через ле